In [8]:
# https://www.ArchRProject.com/bookdown/how-does-archr-make-pseudo-bulk-replicates.html
here::i_am("atac/archR/pseudobulk/1_archR_add_GroupCoverage.R")

source(here::here("settings.R"))
source(here::here("utils.R"))

suppressPackageStartupMessages(library(ArchR))

#rhdf5::h5disableFileLocking()


######################
## Define arguments ##
######################

# p <- ArgumentParser(description='')
# p$add_argument('--metadata',    type="character",    help='metadata file')
# p$add_argument('--archr_directory',    type="character",    help='ArchR directory')
# p$add_argument('--group_by',     type="character",    help='Metadata column to group by')
# p$add_argument('--min_cells',     type="integer",    default=50,   help='Minimum number of cells')
# p$add_argument('--max_cells',     type="integer",    default=1000,   help='Maximum number of cells')
# p$add_argument('--threads',     type="integer",    default=1,    help='Number of threads')

# args <- p$parse_args(commandArgs(TRUE))

## START TEST ##
args = list()
args$archr_directory <- file.path(io$basedir,"processed/atac/archR")
args$metadata <- file.path(io$basedir,"results/atac/archR/qc/sample_metadata_after_qc.txt.gz")
args$group_by <- "celltype_genotype"
args$min_cells <- 50
args$max_cells <- 9999
args$threads <- 1
## END TEST ##
########################
## Load cell metadata ##
########################

cell_metadata.dt <- fread(args$metadata) #%>%
  #.[pass_atacQC==TRUE & sample%in%opts$samples]

# Filter groups by minimum number of cells
# stopifnot(args$group_by%in%colnames(cell_metadata.dt))
# cell_metadata.dt <- cell_metadata.dt[!is.na(cell_metadata.dt[[args$group_by]])]
# cell_metadata.dt <- cell_metadata.dt[!grepl("NA",cell_metadata.dt[[args$group_by]])]
# cell_metadata.dt <- cell_metadata.dt[,N:=.N,by=c(args$group_by)] %>% .[N>=args$min_cells] %>% .[,N:=NULL]

# table(cell_metadata.dt[[args$group_by]])

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/10_Eomes_invitro_gut/code



In [9]:
cell_metadata.dt

cell,barcode,sample,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,alias,day,genotype,⋯,closest.cell,day_celltype,celltype_genotype,TSSEnrichment_atac,ReadsInTSS_atac,PromoterRatio_atac,NucleosomeRatio_atac,nFrags_atac,BlacklistRatio_atac,pass_atacQC
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<int>,<dbl>,<lgl>
1A_Eo_DEG_G9_day3#AAACAGCCAAGTGAAC-1,AAACAGCCAAGTGAAC-1,1A_Eo_DEG_G9_day3,602,1007,0.50,25.02,day3_wt_rep1,D3,wt,⋯,NA,D3-wt,NA-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACAGCCAGCAAGAT-1,AAACAGCCAGCAAGAT-1,1A_Eo_DEG_G9_day3,6110,22923,6.09,10.15,day3_wt_rep1,D3,wt,⋯,cell_85485,D3-wt,Epiblast-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACAGCCAGCACCAT-1,AAACAGCCAGCACCAT-1,1A_Eo_DEG_G9_day3,4201,10794,6.57,7.38,day3_wt_rep1,D3,wt,⋯,cell_88836,D3-wt,Primitive_Streak-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACATGCAACACCTA-1,AAACATGCAACACCTA-1,1A_Eo_DEG_G9_day3,680,1195,2.68,23.51,day3_wt_rep1,D3,wt,⋯,NA,D3-wt,NA-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACATGCAAGTAAGC-1,AAACATGCAAGTAAGC-1,1A_Eo_DEG_G9_day3,3659,8745,5.74,11.92,day3_wt_rep1,D3,wt,⋯,cell_1799,D3-wt,Epiblast-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACATGCAATATACC-1,AAACATGCAATATACC-1,1A_Eo_DEG_G9_day3,3640,8461,3.70,7.21,day3_wt_rep1,D3,wt,⋯,cell_43612,D3-wt,Primitive_Streak-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACATGCAATCCTGA-1,AAACATGCAATCCTGA-1,1A_Eo_DEG_G9_day3,3632,8276,4.60,8.16,day3_wt_rep1,D3,wt,⋯,cell_106685,D3-wt,Epiblast-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACATGCAATGAGGT-1,AAACATGCAATGAGGT-1,1A_Eo_DEG_G9_day3,631,1092,2.11,25.46,day3_wt_rep1,D3,wt,⋯,NA,D3-wt,NA-wt,NA,NA,NA,NA,NA,NA,FALSE
1A_Eo_DEG_G9_day3#AAACATGCACTAAGCC-1,AAACATGCACTAAGCC-1,1A_Eo_DEG_G9_day3,718,1191,1.34,21.58,day3_wt_rep1,D3,wt,⋯,NA,D3-wt,NA-wt,NA,NA,NA,NA,NA,NA,FALSE


In [ ]:


########################
## Load ArchR project ##
########################

# source(here::here("atac/archR/load_archR_project.R"))

setwd(args$archr_directory)

addArchRGenome("mm10")
addArchRThreads(threads = args$threads)

ArchRProject <- loadArchRProject(args$archr_directory)

# Subset
ArchRProject.filt <- ArchRProject[cell_metadata.dt$cell]

###########################
## Update ArchR metadata ##
###########################

cell_metadata.dt.to.archr <- cell_metadata.dt %>% 
  .[cell%in%rownames(ArchRProject.filt)] %>% setkey(cell) %>% .[rownames(ArchRProject.filt)] %>%
  as.data.frame() %>% tibble::column_to_rownames("cell")

stopifnot(all(rownames(cell_metadata.dt.to.archr) == rownames(getCellColData(ArchRProject.filt))))
ArchRProject.filt <- addCellColData(
  ArchRProject.filt,
  data = cell_metadata.dt.to.archr[[args$group_by]],
  name = args$group_by,
  cells = rownames(cell_metadata.dt.to.archr),
  force = TRUE
)

# print cell numbers
table(getCellColData(ArchRProject,"Sample")[[1]])
table(getCellColData(ArchRProject,args$group_by)[[1]])

#########################
## Add Group Coverages ##
#########################

# Check if group Coverages already exist
# ArchRProject@projectMetadata$GroupCoverages

# This function will merge cells within each designated cell group for the generation of pseudo-bulk replicates 
# and then merge these replicates into a single insertion coverage file.
# Output: creates files in archR/GroupCoverages/celltype: [X]._.Rep[Y].insertions.coverage.h5
ArchRProject.filt <- addGroupCoverages(ArchRProject.filt, 
  groupBy = args$group_by,
  useLabels = FALSE,  # do not use sample information
  minCells = args$min_cells,
  maxCells = args$max_cells,
  force = TRUE
)

##########
## Save ##
##########

saveRDS(ArchRProject.filt@projectMetadata, file.path(args$archr_directory,"projectMetadata.rds"))
#saveArchRProject(ArchRProject)